In [ ]:
%%capture
!pip install git+https://github.com/PytorchLightning/lightning-bolts.git@master --upgrade

In [ ]:
import math

import torch
from filelock import FileLock
from torch.nn import functional as F
from torchmetrics import Accuracy
import pytorch_lightning as pl
from pl_bolts.datamodules.mnist_datamodule import MNISTDataModule
import os
from ray.tune.integration.pytorch_lightning import TuneReportCallback

from ray import air, tune


class LightningMNISTClassifier(pl.LightningModule):
    def __init__(self, config, data_dir=None):
        super(LightningMNISTClassifier, self).__init__()

        self.data_dir = data_dir or os.getcwd()
        self.lr = config["lr"]
        layer_1, layer_2 = config["layer_1"], config["layer_2"]
        self.batch_size = config["batch_size"]

        # mnist images are (1, 28, 28) (channels, width, height)
        self.layer_1 = torch.nn.Linear(28 * 28, layer_1)
        self.layer_2 = torch.nn.Linear(layer_1, layer_2)
        self.layer_3 = torch.nn.Linear(layer_2, 10)
        self.accuracy = Accuracy()

    def forward(self, x):
        batch_size, channels, width, height = x.size()
        x = x.view(batch_size, -1)
        x = self.layer_1(x)
        x = torch.relu(x)
        x = self.layer_2(x)
        x = torch.relu(x)
        x = self.layer_3(x)
        x = torch.log_softmax(x, dim=1)
        return x

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)

    def training_step(self, train_batch, batch_idx):
        x, y = train_batch
        logits = self.forward(x)
        loss = F.nll_loss(logits, y)
        acc = self.accuracy(logits, y)
        self.log("ptl/train_loss", loss)
        self.log("ptl/train_accuracy", acc)
        return loss

    def validation_step(self, val_batch, batch_idx):
        x, y = val_batch
        logits = self.forward(x)
        loss = F.nll_loss(logits, y)
        acc = self.accuracy(logits, y)
        return {"val_loss": loss, "val_accuracy": acc}

    def validation_epoch_end(self, outputs):
        avg_loss = torch.stack([x["val_loss"] for x in outputs]).mean()
        avg_acc = torch.stack([x["val_accuracy"] for x in outputs]).mean()
        self.log("ptl/val_loss", avg_loss)
        self.log("ptl/val_accuracy", avg_acc)


def train_mnist_tune(config, num_epochs=10, num_gpus=0):
    data_dir = os.path.abspath("./data")
    model = LightningMNISTClassifier(config, data_dir)
    with FileLock(os.path.expanduser("~/.data.lock")):
        dm = MNISTDataModule(
            data_dir=data_dir, num_workers=1, batch_size=config["batch_size"]
        )
    metrics = {"loss": "ptl/val_loss", "acc": "ptl/val_accuracy"}
    trainer = pl.Trainer(
        max_epochs=num_epochs,
        # If fractional GPUs passed in, convert to int.
        gpus=math.ceil(num_gpus),
        enable_progress_bar=False,
        callbacks=[TuneReportCallback(metrics, on="validation_end")],
    )
    trainer.fit(model, dm)


def tune_mnist(num_samples=10, num_epochs=10, gpus_per_trial=0):
    config = {
        "layer_1": tune.choice([32, 64, 128]),
        "layer_2": tune.choice([64, 128, 256]),
        "lr": tune.loguniform(1e-4, 1e-1),
        "batch_size": tune.choice([32, 64, 128]),
    }

    trainable = tune.with_parameters(
        train_mnist_tune, num_epochs=num_epochs, num_gpus=gpus_per_trial
    )
    tuner = tune.Tuner(
        tune.with_resources(trainable, resources={"cpu": 2, "gpu": gpus_per_trial}),
        tune_config=tune.TuneConfig(
            metric="loss",
            mode="min",
            num_samples=num_samples,
        ),
        run_config=air.RunConfig(
            name="tune_mnist",
        ),
        param_space=config,
    )
    results = tuner.fit()

    print("Best hyperparameters found were: ", results.get_best_result().config)


if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--smoke-test", action="store_true", help="Finish quickly for testing"
    )
    args, _ = parser.parse_known_args()

    if args.smoke_test:
        tune_mnist(num_samples=1, num_epochs=1, gpus_per_trial=0)
    else:
        tune_mnist(num_samples=10, num_epochs=10, gpus_per_trial=0.5)

/usr/local/lib/python3.10/dist-packages/pl_bolts/models/self_supervised/amdim/amdim_module.py:34: UnderReviewWarning: The feature generate_power_seq is currently marked under review. The compatibility with other Lightning projects is not guaranteed and API may change at any time. The API and functionality may change without warning in future releases. More details: https://lightning-bolts.readthedocs.io/en/latest/stability.html
  "lr_options": generate_power_seq(LEARNING_RATE_CIFAR, 11),
/usr/local/lib/python3.10/dist-packages/pl_bolts/models/self_supervised/amdim/amdim_module.py:92: UnderReviewWarning: The feature FeatureMapContrastiveTask is currently marked under review. The compatibility with other Lightning projects is not guaranteed and API may change at any time. The API and functionality may change without warning in future releases. More details: https://lightning-bolts.readthedocs.io/en/latest/stability.html
  contrastive_task: Union[FeatureMapContrastiveTask] = FeatureMapCon

== Status ==
Current time: 2023-06-21 20:06:36 (running for 00:00:00.26)
Using FIFO scheduling algorithm.
Logical resource usage: 0/2 CPUs, 0/1 GPUs
Result logdir: /root/ray_results/tune_mnist
Number of trials: 10/10 (10 PENDING)
+------------------------------+----------+-------+--------------+-----------+-----------+-------------+
| Trial name                   | status   | loc   |   batch_size |   layer_1 |   layer_2 |          lr |
|------------------------------+----------+-------+--------------+-----------+-----------+-------------|
| train_mnist_tune_20547_00000 | PENDING  |       |           32 |        32 |       256 | 0.0144678   |
| train_mnist_tune_20547_00001 | PENDING  |       |           64 |        32 |       128 | 0.0143091   |
| train_mnist_tune_20547_00002 | PENDING  |       |           64 |        32 |       256 | 0.0100861   |
| train_mnist_tune_20547_00003 | PENDING  |       |          128 |        32 |       128 | 0.000141841 |
| train_mnist_tune_20547_00004 | PE

(pid=7370) /usr/local/lib/python3.10/dist-packages/pl_bolts/models/self_supervised/amdim/amdim_module.py:34: UnderReviewWarning: The feature generate_power_seq is currently marked under review. The compatibility with other Lightning projects is not guaranteed and API may change at any time. The API and functionality may change without warning in future releases. More details: https://lightning-bolts.readthedocs.io/en/latest/stability.html
(pid=7370)   "lr_options": generate_power_seq(LEARNING_RATE_CIFAR, 11),
(pid=7370) /usr/local/lib/python3.10/dist-packages/pl_bolts/models/self_supervised/amdim/amdim_module.py:92: UnderReviewWarning: The feature FeatureMapContrastiveTask is currently marked under review. The compatibility with other Lightning projects is not guaranteed and API may change at any time. The API and functionality may change without warning in future releases. More details: https://lightning-bolts.readthedocs.io/en/latest/stability.html
(pid=7370)   contrastive_task: Unio

(train_mnist_tune pid=7370) Downloading http://yann.lecun.com/exdb/mnist/train-images-idx3-ubyte.gz
(train_mnist_tune pid=7370) Downloading http://yann.lecun.com/exdb/mnist/train-images-idx3-ubyte.gz to /root/ray_results/tune_mnist/train_mnist_tune_20547_00000_0_batch_size=32,layer_1=32,layer_2=256,lr=0.0145_2023-06-21_20-06-36/data/MNIST/raw/train-images-idx3-ubyte.gz


100%|██████████| 9912422/9912422 [00:00<00:00, 132765274.51it/s]


...

(train_mnist_tune pid=7370) Extracting /root/ray_results/tune_mnist/train_mnist_tune_20547_00000_0_batch_size=32,layer_1=32,layer_2=256,lr=0.0145_2023-06-21_20-06-36/data/MNIST/raw/t10k-images-idx3-ubyte.gz to /root/ray_results/tune_mnist/train_mnist_tune_20547_00000_0_batch_size=32,layer_1=32,layer_2=256,lr=0.0145_2023-06-21_20-06-36/data/MNIST/raw
(train_mnist_tune pid=7370) 
(train_mnist_tune pid=7370) Downloading http://yann.lecun.com/exdb/mnist/t10k-labels-idx1-ubyte.gz
(train_mnist_tune pid=7370) Downloading http://yann.lecun.com/exdb/mnist/t10k-labels-idx1-ubyte.gz to /root/ray_results/tune_mnist/train_mnist_tune_20547_00000_0_batch_size=32,layer_1=32,layer_2=256,lr=0.0145_2023-06-21_20-06-36/data/MNIST/raw/t10k-labels-idx1-ubyte.gz
(train_mnist_tune pid=7370) Extracting /root/ray_results/tune_mnist/train_mnist_tune_20547_00000_0_batch_size=32,layer_1=32,layer_2=256,lr=0.0145_2023-06-21_20-06-36/data/MNIST/raw/t10k-labels-idx1-ubyte.gz to /root/ray_results/tune_mnist/train_mnist

100%|██████████| 4542/4542 [00:00<00:00, 24613086.26it/s]
(train_mnist_tune pid=7370) Missing logger folder: /root/ray_results/tune_mnist/train_mnist_tune_20547_00000_0_batch_size=32,layer_1=32,layer_2=256,lr=0.0145_2023-06-21_20-06-36/lightning_logs
(train_mnist_tune pid=7370) LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
(train_mnist_tune pid=7370) 
(train_mnist_tune pid=7370)   | Name     | Type     | Params
(train_mnist_tune pid=7370) --------------------------------------
(train_mnist_tune pid=7370) 0 | layer_1  | Linear   | 25.1 K
(train_mnist_tune pid=7370) 1 | layer_2  | Linear   | 8.4 K 
(train_mnist_tune pid=7370) 2 | layer_3  | Linear   | 2.6 K 
(train_mnist_tune pid=7370) 3 | accuracy | Accuracy | 0     
(train_mnist_tune pid=7370) --------------------------------------
(train_mnist_tune pid=7370) 36.1 K    Trainable params
(train_mnist_tune pid=7370) 0         Non-trainable params
(train_mnist_tune pid=7370) 36.1 K    Total params
(train_mnist_tune pid=7370) 0.145     Total es

== Status ==
Current time: 2023-06-21 20:06:51 (running for 00:00:15.37)
Using FIFO scheduling algorithm.
Logical resource usage: 2.0/2 CPUs, 0.5/1 GPUs
Result logdir: /root/ray_results/tune_mnist
Number of trials: 10/10 (9 PENDING, 1 RUNNING)
+------------------------------+----------+------------------+--------------+-----------+-----------+-------------+
| Trial name                   | status   | loc              |   batch_size |   layer_1 |   layer_2 |          lr |
|------------------------------+----------+------------------+--------------+-----------+-----------+-------------|
| train_mnist_tune_20547_00000 | RUNNING  | 172.28.0.12:7370 |           32 |        32 |       256 | 0.0144678   |
| train_mnist_tune_20547_00001 | PENDING  |                  |           64 |        32 |       128 | 0.0143091   |
| train_mnist_tune_20547_00002 | PENDING  |                  |           64 |        32 |       256 | 0.0100861   |
| train_mnist_tune_20547_00003 | PENDING  |                 

(train_mnist_tune pid=7370) /usr/local/lib/python3.10/dist-packages/ray/tune/trainable/session.py:240: DeprecationWarning: `tune.report` and `tune.checkpoint_dir` APIs are deprecated in Ray 2.0, and is replaced by `ray.air.session`. This will provide an easy-to-use API across Tune session and Data parallel worker sessions.The old APIs will be removed in the future. 
(train_mnist_tune pid=7370)   warnings.warn(


== Status ==
Current time: 2023-06-21 20:07:17 (running for 00:00:40.57)
Using FIFO scheduling algorithm.
Logical resource usage: 2.0/2 CPUs, 0.5/1 GPUs
Current best trial: 20547_00000 with loss=0.34397652745246887 and parameters={'layer_1': 32, 'layer_2': 256, 'lr': 0.01446777576958688, 'batch_size': 32}
Result logdir: /root/ray_results/tune_mnist
Number of trials: 10/10 (9 PENDING, 1 RUNNING)
+------------------------------+----------+------------------+--------------+-----------+-----------+-------------+--------+------------------+----------+---------+
| Trial name                   | status   | loc              |   batch_size |   layer_1 |   layer_2 |          lr |   iter |   total time (s) |     loss |     acc |
|------------------------------+----------+------------------+--------------+-----------+-----------+-------------+--------+------------------+----------+---------|
| train_mnist_tune_20547_00000 | RUNNING  | 172.28.0.12:7370 |           32 |        32 |       256 | 0.014

In [ ]:
!pip install reportlab

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 44.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 74.1 MB/s eta 0:00:00
  Attempting uninstall: pillow
    Found existing installation: Pillow 8.4.0
    Uninstalling Pillow-8.4.0:
      Successfully uninstalled Pillow-8.4.0


In [ ]:
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas

def generate_cv():
    # Create a new PDF canvas
    c = canvas.Canvas("cv.pdf", pagesize=letter)

    # Set font styles
    c.setFont("Helvetica-Bold", 14)
    c.setFont("Helvetica", 12)

    # Write content to the PDF
    c.drawString(100, 750, "John Doe")
    c.drawString(100, 730, "Software Developer")
    c.drawString(100, 710, "123 Main St, City, Postal Code")
    c.drawString(100, 690, "johndoe@email.com")
    c.drawString(100, 670, "Phone: 123-456-7890")

    # Save the PDF
    c.save()

# Call the function to generate the CV
generate_cv()


In [ ]:
%%capture
!pip install lightning

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader
import lightning as L
import os

class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1)
        self.relu = nn.ReLU()
        self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1)
        self.fc1 = nn.Linear(32 * 7 * 7, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.conv2(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        return x

fabric = L.Fabric(accelerator="cuda", devices=1, strategy="ddp")
fabric.launch()
# Set device (CPU or GPU)
device = fabric.device

# Create the "dataset" directory if it doesn't exist
os.makedirs("dataset", exist_ok=True)

# Set the root path for the MNIST dataset
root_path = "dataset"

# Load the MNIST dataset
with fabric.rank_zero_first():
    train_dataset = MNIST(root=root_path, train=True, download=True, transform=ToTensor())
    test_dataset = MNIST(root=root_path, train=False, download=False, transform=ToTensor())
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

train_loader, test_loader = fabric.setup_dataloaders(train_loader, test_loader)

# Create the CNN model
model = CNN().to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

model, optimizer = fabric.setup(model, optimizer)

# Training loop
for epoch in range(10):
    running_loss = 0.0
    for i, (inputs, labels) in enumerate(train_loader, 0):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        # loss.backward()
        fabric.backward(loss)
        optimizer.step()

        running_loss += loss.item()
        if i % 200 == 199:  # Print every 200 mini-batches
            print(f"[Epoch {epoch+1}, Batch {i+1}] Loss: {running_loss/200:.3f}")
            running_loss = 0.0

print("Training finished.")

# Test the model
correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Accuracy on the test set: {accuracy:.2f}%")

RuntimeError: ignored

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

Mounted at /content/gdrive


In [ ]:
import os
os.chdir("/content/gdrive/My Drive/TL_A/DeepKale/experiments_0.2/")

In [ ]:
root_dir = '/content/gdrive/My Drive/TL_A/DeepKale/experiments_0.2/'

In [ ]:
root_data_dir = root_dir + 'data/'
raw_dir = root_data_dir + 'raw/ACN/'
preprocessed_dir = root_data_dir + 'preprocessed/ACN/'

In [ ]:
import json

In [ ]:
file_path = os.path.join(raw_dir, 'acndata_sessions.json')
with open(file_path, 'r') as file:
    data = json.load(file)

In [ ]:
import pandas as pd


In [ ]:
df = pd.DataFrame(data['_items'])
df

,_id,clusterID,connectionTime,disconnectTime,doneChargingTime,kWhDelivered,sessionID,siteID,spaceID,stationID,timezone,userID,userInputs
0,5e4de11ff9af8b5cdd578316,0102,"Mon, 03 Feb 2020 15:47:02 GMT","Tue, 04 Feb 2020 02:19:51 GMT","Mon, 03 Feb 2020 17:46:26 GMT",6.004,19_102_260_1638_2020-02-03 15:47:02.362333,0019,04,19-102-260-1638,America/Los_Angeles,None,None
1,5e4de11ff9af8b5cdd578317,0102,"Mon, 03 Feb 2020 18:21:21 GMT","Tue, 04 Feb 2020 02:23:02 GMT","Mon, 03 Feb 2020 22:13:11 GMT",18.472,19_102_260_1640_2020-02-03 18:21:21.413397,0019,08,19-102-260-1640,America/Los_Angeles,None,None
2,5e4de11ff9af8b5cdd578318,0102,"Mon, 03 Feb 2020 18:34:05 GMT","Tue, 04 Feb 2020 00:20:25 GMT","Mon, 03 Feb 2020 20:19:42 GMT",4.755,19_102_260_1636_2020-02-03 18:34:05.470756,0019,06,19-102-260-1636,America/Los_Angeles,None,None
3,5e4de11ff9af8b5cdd578319,0102,"Mon, 03 Feb 2020 19:13:10 GMT","Tue, 04 Feb 2020 01:35:13 GMT","Tue, 04 Feb 2020 01:35:05 GMT",28.913,19_102_260_1634_2020-02-03 19:13:09.783956,0019,03,19-102-260-1634,America/Los_Angeles,None,None
4,5e4de11ff9af8b5cdd57831a,0102,"Mon, 03 Feb 2020 21:23:53 GMT","Tue, 04 Feb 2020 01:25:19 GMT","Tue, 04 Feb 2020 01:25:10 GMT",13.429,19_102_260_1639_2020-02-03 21:23:52.817193,0019,05,19-102-260-1639,America/Los_Angeles,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...
616,614fbf0df9af8b6e4f11442d,0102,"Thu, 09 Sep 2021 21:11:44 GMT","Fri, 10 Sep 2021 03:47:58 GMT","Fri, 10 Sep 2021 03:47:33 GMT",19.342,19_102_260_1640_2021-09-09 21:11:26.620176,0019,08,19-102-260-1640,America/Los_Angeles,000006140,"[{'WhPerMile': 258, 'kWhRequested': 72.24, 'mi..."
617,6151108ff9af8b70789b4fed,0102,"Fri, 10 Sep 2021 15:33:08 GMT","Fri, 10 Sep 2021 23:53:29 GMT","Fri, 10 Sep 2021 22:49:37 GMT",33.186,19_102_260_1634_2021-09-10 15:32:59.042125,0019,03,19-102-260-1634,America/Los_Angeles,000006620,"[{'WhPerMile': 1428, 'kWhRequested': 114.24, '..."
618,6151108ff9af8b70789b4fee,0102,"Fri, 10 Sep 2021 21:12:26 GMT","Sat, 11 Sep 2021 04:25:47 GMT","Sat, 11 Sep 2021 04:25:34 GMT",19.894,19_102_260_1640_2021-09-10 21:12:15.541083,0019,08,19-102-260-1640,America/Los_Angeles,000006140,"[{'WhPerMile': 258, 'kWhRequested': 72.24, 'mi..."
619,61550514f9af8b76948f5921,0102,"Mon, 13 Sep 2021 17:35:32 GMT","Tue, 14 Sep 2021 00:26:25 GMT","Tue, 14 Sep 2021 00:26:05 GMT",26.688,19_102_260_1634_2021-09-13 17:35:24.605831,0019,03,19-102-260-1634,America/Los_Angeles,000006620,"[{'WhPerMile': 1428, 'kWhRequested': 114.24, '..."


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
df['connectionTime'] = pd.to_datetime(df['connectionTime'],utc=False,errors='ignore')

In [ ]:
import plotly.express as px
fig = px.line(df, x='connectionTime', y='kWhDelivered', title='Plot of kWh Values')
fig.update_layout(xaxis_title='X Axis Label', yaxis_title='kWh Value')

fig.show()

In [2]:
from functools import partial
from itertools import chain

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

%matplotlib inline
np.random.seed(1)

In [4]:
def f(x):
    """The function to predict."""
    return x * np.sin(x)

#----------------------------------------------------------------------
#  First the noiseless case
X = np.atleast_2d(np.random.uniform(0, 10.0, size=100)).T
X = X.astype(np.float32)

# Observations
y = f(X).ravel()

dy = 1.5 + 1.0 * np.random.random(y.shape)
noise = np.random.normal(0, dy)
y += noise
y = y.astype(np.float32)

# Mesh the input space for evaluations of the real function, the prediction and
# its MSE
xx = np.atleast_2d(np.linspace(0, 10, 1000)).T
xx = xx.astype(np.float32)

X.shape, y.shape, xx.shape

((100, 1), (100,), (1000, 1))

In [ ]:
class q_model_simplified(nn.Module):
    def __init__(self,
                 quantiles,
                 in_shape=1,
                 dropout=0.5):
        super().__init__()
        self.quantiles = quantiles
        self.num_quantiles = len(quantiles)

        self.in_shape = in_shape
        self.out_shape = len(quantiles)
        self.dropout = dropout
        self.build_model()
        self.init_weights()

    def build_model(self):
        self.model = nn.Sequential(
            nn.Linear(self.in_shape, 64),
            nn.ReLU(),
            # nn.BatchNorm1d(64),
            nn.Dropout(self.dropout),
            nn.Linear(64, 128),
            nn.ReLU(),
            # nn.BatchNorm1d(128),
            nn.Dropout(self.dropout),
            nn.Linear(128, self.out_shape)
        )

    def init_weights(self):
        for m in self.model:
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        return self.model(x)

In [ ]:
class QuantileLoss(nn.Module):
    def __init__(self, quantiles):
        super().__init__()
        self.quantiles = quantiles

    def forward(self, preds, target):
        assert not target.requires_grad
        assert preds.size(0) == target.size(0)
        losses = []
        for i, q in enumerate(quantiles):
            errors = target - preds[:, i]
            losses.append(torch.max((q-1) * errors, q * errors).unsqueeze(1))
        loss = torch.mean(torch.sum(torch.cat(losses, dim=1), dim=1))
        return loss

In [ ]:
class Learner:
    def __init__(self, model, optimizer_class, loss_func, device='cpu'):
        self.model = model.to(device)
        self.optimizer = optimizer_class(self.model.parameters())
        self.loss_func = loss_func.to(device)
        self.device = device
        self.loss_history = []

    def fit(self, x, y, epochs, batch_size):
        self.model.train()
        for e in range(epochs):
            shuffle_idx = np.arange(x.shape[0])
            np.random.shuffle(shuffle_idx)
            x = x[shuffle_idx]
            y = y[shuffle_idx]
            epoch_losses = []
            for idx in range(0, x.shape[0], batch_size):
                self.optimizer.zero_grad()
                batch_x = torch.from_numpy(
                    x[idx : min(idx + batch_size, x.shape[0]),:]
                ).float().to(self.device).requires_grad_(False)
                batch_y = torch.from_numpy(
                    y[idx : min(idx + batch_size, y.shape[0])]
                ).float().to(self.device).requires_grad_(False)
                preds = self.model(batch_x)
                loss = loss_func(preds, batch_y)
                loss.backward()
                self.optimizer.step()
                epoch_losses.append(loss.cpu().detach().numpy())
            epoch_loss =  np.mean(epoch_losses)
            self.loss_history.append(epoch_loss)
            if (e+1) % 500 == 0:
                print("Epoch {}: {}".format(e+1, epoch_loss))

    def predict(self, x, mc=False):
        if mc:
            self.model.train()
        else:
            self.model.eval()
        return self.model(torch.from_numpy(x).to(self.device).requires_grad_(False)).cpu().detach().numpy()

In [ ]:
class q_model(nn.Module):
    def __init__(self,
                 quantiles,
                 in_shape=1,
                 dropout=0.5):
        super().__init__()
        self.quantiles = quantiles
        self.num_quantiles = len(quantiles)

        self.in_shape = in_shape
        self.out_shape = len(quantiles)
        self.dropout = dropout
        self.build_model()
        self.init_weights()

    def build_model(self):
        self.base_model = nn.Sequential(
            nn.Linear(self.in_shape, 64),
            nn.ReLU(),
            # nn.BatchNorm1d(64),
            nn.Dropout(self.dropout),
            nn.Linear(64, 64),
            nn.ReLU(),
            # nn.BatchNorm1d(64),
            nn.Dropout(self.dropout),
        )
        final_layers = [
            nn.Linear(64, 1) for _ in range(len(self.quantiles))
        ]
        self.final_layers = nn.ModuleList(final_layers)

    def init_weights(self):
        for m in chain(self.base_model, self.final_layers):
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        tmp_ = self.base_model(x)
        return torch.cat([layer(tmp_) for layer in self.final_layers], dim=1)

In [ ]:
# Instantiate model
quantiles = [.05, .5, .95]
model = q_model(quantiles, dropout=0.1)
loss_func = QuantileLoss(quantiles)
learner = Learner(model, partial(torch.optim.Adam, weight_decay=1e-6), loss_func)

In [ ]:
epochs = 10000
learner.fit(X, y, epochs, batch_size=10)

Epoch 500: 1.6647026538848877
Epoch 1000: 1.5970925092697144
Epoch 1500: 1.6138505935668945
Epoch 2000: 1.3193504810333252
Epoch 2500: 1.2481276988983154
Epoch 3000: 1.3369371891021729
Epoch 3500: 1.279969573020935
Epoch 4000: 1.5422189235687256
Epoch 4500: 1.2010793685913086
Epoch 5000: 1.2263308763504028
Epoch 5500: 1.2219606637954712
Epoch 6000: 1.2560417652130127
Epoch 6500: 1.2269655466079712
Epoch 7000: 1.2225029468536377
Epoch 7500: 1.3045597076416016
Epoch 8000: 1.1351505517959595
Epoch 8500: 1.1812419891357422
Epoch 9000: 1.2562779188156128
Epoch 9500: 1.1543307304382324
Epoch 10000: 1.1435974836349487
